In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import sqlite3

con = sqlite3.connect("../data/social.db")     # creates the file if it doesn't exist

tables = {
    "posts": "../data/social_posts.csv",
    "brands": "../data/brands.csv",
    "orders": "../data/gate_orders.csv",
    "customers": "../data/gate_customers.csv",
}
for name, path in tables.items():
    pd.read_csv(path).to_sql(name, con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, con)

q("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,toy
1,posts
2,brands
3,orders
4,customers


import pandas as pd, sqlite3
con = sqlite3.connect("../data/social.db")
def q(sql): return pd.read_sql(sql, con)

pandas	SQL
df[["a", "b"]]	SELECT a, b FROM df
df[df["a"] > 5]	WHERE a > 5
df.sort_values("a", ascending=False)	ORDER BY a DESC
.head(5)	LIMIT 5
df["a"].unique()	SELECT DISTINCT a
len(df)	SELECT COUNT(*)
df["a"].isna()	a IS NULL

# SELECT post_id, brand, likes      -- which columns
# FROM posts                         -- which table

q("""
SELECT post_id, brand, likes
FROM posts
""")

SELECT post_id, likes / 1000.0 AS likes_k
FROM posts

SELECT 7 / 2         -- 3    ← both whole numbers → whole-number answer
SELECT 7 / 2.0       -- 3.5  ✓

SELECT likes * 100.0 / followers AS like_rate FROM posts
# ROUND(x, 2) rounds, like pandas .round(2).

SELECT post_id, likes
FROM posts
WHERE platform = 'TikTok' AND likes > 5000
# Text goes in single quotes: 'TikTok'
# One = for “equals” (not ==)
# Not equal: != or <>
# AND, OR, NOT are written as words (like query, not like masks)

Operator	Example	Meaning
IN	brand IN ('Glowly', 'FitFuel')	matches any in the list (pandas isin)
NOT IN	platform NOT IN ('TikTok')	none of the list
BETWEEN	likes BETWEEN 1000 AND 2000	both ends included
LIKE	brand LIKE '%nest%'	text pattern: % = “anything”
IS NULL	comments IS NULL	missing
IS NOT NULL	comments IS NOT NULL	not missing

# = is case-sensitive: brand = 'glowly' finds nothing, because the data says 'Glowly'. (LIKE is case-insensitive in SQLite.)

# Dates: SQLite stores your dates as text like '2026-03-15'. Because that format runs year → month → day, text comparison still works:

WHERE date BETWEEN '2026-03-01' AND '2026-03-31'

WHERE platform = 'TikTok' OR platform = 'Facebook' AND likes > 2000
-- read as:  TikTok   OR   (Facebook AND likes > 2000)   → 105 rows ✗
WHERE (platform = 'TikTok' OR platform = 'Facebook') AND likes > 2000
-- ✓ 36 rows

WHERE comments = NULL      -- 0 rows, always ✗
WHERE comments IS NULL     -- 10 rows ✓

SELECT post_id, brand, likes
FROM posts
ORDER BY likes DESC          -- DESC = highest first; ASC (default) = lowest first
LIMIT 5

# Several columns: ORDER BY brand ASC, likes DESC sorts by brand, then by likes within each brand.
# In SQLite, NULLs come first when sorting ascending.

SELECT DISTINCT platform FROM posts            -- unique values
SELECT DISTINCT brand, platform FROM posts     -- unique COMBINATIONS
SELECT COUNT(*) FROM posts                     -- number of rows
SELECT COUNT(comments) FROM posts              -- non-NULL values only (like pandas .count())
SELECT COUNT(DISTINCT brand) FROM posts        -- number of unique values (like nunique)

SELECT → FROM → WHERE → ORDER BY → LIMIT

Writing order vs running order

You write a query in this order: SELECT → FROM → WHERE → ORDER BY → LIMIT.

The database runs it in this order:

FROM: which table
WHERE: which rows
SELECT: which columns / calculations
ORDER BY: sort
LIMIT: cut

This explains why ORDER BY can use an alias from SELECT (it runs after it), but in most databases WHERE can’t (it runs before SELECT exists). SQLite is lenient about this, but don’t rely on it, because interviews use other databases.

## 3.1 Cheat sheet: SELECT basics

| I want… | SQL |
|---|---|
| some columns | `SELECT a, b FROM t` |
| all columns | `SELECT * FROM t` |
| new column + name | `SELECT a * 100.0 / b AS rate FROM t` |
| filter | `WHERE a > 5 AND b = 'x'` |
| mix AND + OR | `WHERE (a = 'x' OR a = 'y') AND b > 5` |
| in a list | `WHERE a IN ('x', 'y')` |
| range (inclusive) | `WHERE a BETWEEN 10 AND 20` |
| text pattern | `WHERE a LIKE '%nest%'` |
| missing / not missing | `WHERE a IS NULL` / `IS NOT NULL` |
| sort | `ORDER BY a DESC, b ASC` |
| top N | `ORDER BY a DESC LIMIT 5` |
| unique values | `SELECT DISTINCT a FROM t` |
| count rows / non-null / unique | `COUNT(*)` / `COUNT(a)` / `COUNT(DISTINCT a)` |
| round | `ROUND(x, 2)` |

**Rules**
- Text in 'single quotes'; one = for equals; = is case-sensitive
- int / int = int → use 100.0 or 1.0 *
- = NULL never works → IS NULL
- AND before OR → brackets
- Runs: FROM → WHERE → SELECT → ORDER BY → LIMIT

In [2]:
pd.DataFrame({
    "name":  ["a", "b", "c", "d", "e"],
    "team":  ["x", "y", "x", "y", "x"],
    "score": [5, 12, None, 20, 8],
    "city":  ["UK", "US", "UK", None, "US"],
}).to_sql("toy", con, index=False, if_exists="replace")

5

In [3]:
q("SELECT COUNT(*) FROM toy") #5
q("SELECT COUNT(score) FROM toy") #4
q("SELECT name FROM toy WHERE score > 6 ORDER BY score DESC") #d, b, e
q("SELECT name FROM toy WHERE score = NULL") #nothing
q("SELECT name FROM toy WHERE score IS NULL") #c
q("SELECT DISTINCT team FROM toy") #: how many rows? x, y
q("SELECT COUNT(DISTINCT city) FROM toy") # 2--uk, us
q("SELECT name FROM toy WHERE team = 'x' OR team = 'y' AND score > 10")
# : careful! #d,b, a,c,e
q("SELECT 7 / 2, 7 / 2.0") # 3, 3.5
q("SELECT name FROM toy ORDER BY score LIMIT 2") #c, a 


,COUNT(*)
0,5


,COUNT(score)
0,4


,name
0,d
1,b
2,e


,name


,name
0,c


,team
0,x
1,y


,COUNT(DISTINCT city)
0,2


,name
0,a
1,b
2,c
3,d
4,e


,7 / 2,7 / 2.0
0,3,3.5


,name
0,c
1,a


In [4]:
# B1. Look around

# (a) All columns, first 5 rows.
q("SELECT * FROM posts LIMIT 5")
# (b) Count all rows.
q("SELECT COUNT(*) FROM posts")
# (c) Count the unique brands.
q("SELECT COUNT(DISTINCT brand) FROM posts")

,post_id,date,platform,brand,post_type,followers,likes,comments,shares,sponsored,caption_length
0,P0001,2026-01-01,Instagram,Glowly,image,82837,631,34.0,16.0,0,158
1,P0002,2026-01-02,TikTok,HomeNest,image,27506,485,12.0,12.0,0,311
2,P0003,2026-01-02,Instagram,FitFuel,video,61341,3158,88.0,52.0,0,277
3,P0004,2026-01-03,Instagram,SnackBox,image,118210,1612,75.0,27.0,0,236
4,P0005,2026-01-04,TikTok,SnackBox,image,122826,1387,70.0,15.0,0,121


,COUNT(*)
0,300


,COUNT(DISTINCT brand)
0,5


In [5]:
# B2. Basic filters

# (a) Count the TikTok posts with more than 5000 likes.
q("SELECT COUNT(*) FROM posts WHERE likes>5000 AND platform= 'TikTok'")
# (b) Count the Instagram videos with more than 2000 likes.
q("SELECT COUNT(*) FROM posts WHERE likes>2000 AND platform= 'Instagram' AND post_type= 'video'")

,COUNT(*)
0,8


,COUNT(*)
0,13


In [6]:
# B3. Top posts

# (a) Show post_id, brand, likes for the 5 most-liked posts.
q("SELECT post_id, brand, likes FROM posts ORDER BY likes DESC LIMIT 5")
# (b) Show the single least-liked post (post_id, likes).
q("SELECT post_id, likes FROM posts ORDER BY likes ASC LIMIT 1")

,post_id,brand,likes
0,P0019,SnackBox,13577
1,P0103,SnackBox,12052
2,P0129,SnackBox,9030
3,P0277,SnackBox,6133
4,P0210,Glowly,5761


,post_id,likes
0,P0082,31


In [7]:
# B4. Lists and ranges

# (a) Count posts from Glowly or FitFuel, using IN.
q("SELECT COUNT(*) FROM posts WHERE brand IN ('Glowly', 'FitFuel')")
# (b) Count posts with likes between 1000 and 2000.
q("SELECT COUNT(*) FROM posts WHERE likes BETWEEN 1000 AND 2000")
# (c) Count posts that are not on TikTok, using NOT IN.
q("SELECT COUNT(*) FROM posts WHERE platform NOT IN ('TikTok')")

,COUNT(*)
0,118


,COUNT(*)
0,86


,COUNT(*)
0,196


In [8]:
# B5. Text and dates

# (a) Count posts whose brand contains "nest" (any case), using LIKE.
q("SELECT COUNT(*) FROM posts WHERE brand LIKE '%nest%' ")
# (b) Count posts in March 2026, using BETWEEN.
q("SELECT COUNT(*) FROM posts WHERE date BETWEEN '2026-03-01' AND '2026-03-31' ")

,COUNT(*)
0,66


,COUNT(*)
0,61


In [9]:
# B6. Missing values

# (a) Count posts with missing comments.
q("SELECT COUNT(*) FROM posts WHERE comments IS NULL")
# (b) Run the same with = NULL. Comment: why the difference?
q("SELECT COUNT(*) FROM posts WHERE comments = NULL") 
# NULL isn’t equal to anything, not even another NULL, so = NULL is never true.

,COUNT(*)
0,10


,COUNT(*)
0,0


In [10]:
# B7. Calculated column

# (a) like_rate = likes × 100 / followers, rounded to 2.
q("SELECT ROUND(likes * 100.0 / followers, 2) AS like_rate FROM posts")
# (b) Show post_id, likes, followers, like_rate for the top 3 by like_rate.
q("SELECT post_id, likes, followers, ROUND(likes * 100.0 / followers, 2) AS like_rate FROM posts ORDER BY like_rate DESC LIMIT 3")
# (c) Run SELECT likes / followers FROM posts LIMIT 3. Comment: why all 0?
q("SELECT likes / followers  FROM posts LIMIT 3") #likes<followers so whole division returns 0

,like_rate
0,0.76
1,1.76
2,5.15
3,1.36
4,1.13
...,...
295,0.55
296,0.71
297,1.22
298,3.26


,post_id,likes,followers,like_rate
0,P0019,13577,124109,10.94
1,P0103,12052,116441,10.35
2,P0251,3320,40945,8.11


,likes / followers
0,0
1,0
2,0


In [11]:
# B8. Brackets matter

# (a) Count posts on (TikTok or Facebook) with more than 2000 likes, with brackets.
q("SELECT COUNT(*) FROM posts WHERE (platform = 'TikTok' OR platform ='Facebook') AND likes > 2000")
# (b) Run it without brackets.
q("SELECT COUNT(*) FROM posts WHERE platform = 'TikTok' OR platform ='Facebook' AND likes>2000 ")
# (c) Comment: why are the numbers different?
#AND runs before OR

,COUNT(*)
0,35


,COUNT(*)
0,104


In [12]:
# B9. Combine everything

# (a) Sponsored posts (sponsored = 1) from 1 June 2026 onwards.
q("SELECT post_id, brand, likes FROM posts WHERE sponsored=1 AND date >= '2026-06-01' ORDER BY likes DESC LIMIT 3")
# (b) Show post_id, brand, likes.
# (c) Top 3 by likes.

,post_id,brand,likes
0,P0269,Glowly,4440
1,P0299,SnackBox,4107
2,P0257,SnackBox,3174


In [13]:
q("SELECT post_id FROM posts WHERE platform = 'TikTok' AND likes >= 5000")
# D1: "TikTok" in double quotes means a column name in SQL, and => isn’t an operator (it’s >=).

,post_id
0,P0019
1,P0043
2,P0071
3,P0075
4,P0103
5,P0129
6,P0210
7,P0230


In [14]:
q("SELECT COUNT(*) FROM posts WHERE brand = 'Glowly'")
# D2: = is case-sensitive.

,COUNT(*)
0,62


In [16]:
q("SELECT post_id, likes FROM posts ORDER BY likes DESC LIMIT 3")

,post_id,likes
0,P0019,13577
1,P0103,12052
2,P0129,9030
